In [1]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights, vgg16_bn, VGG16_BN_Weights
from torchsummary import summary
import numpy as np
from cvat_sdk import make_client
from cvat_sdk.pytorch import ProjectVisionDataset
from torchvision import transforms
from tqdm.notebook import tqdm

In [2]:
DETECTION_CLASSES = 6

class ResNet50(nn.Module):
    def __init__(self):
        super(ResNet50, self).__init__()
        

        self.conv = resnet50(weights=ResNet50_Weights.DEFAULT)
        for param in self.conv.parameters():
            param.requires_grad = False

        self.fc = nn.Sequential(
            nn.Linear(in_features = 1000, out_features = 1000, bias=True),
            nn.BatchNorm1d(1000, momentum = 0.5),
            nn.LeakyReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features = 1000, out_features = 500, bias=True),
            nn.BatchNorm1d(500, momentum = 0.5),
            nn.LeakyReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features = 500, out_features = 500, bias=True),
            nn.BatchNorm1d(500, momentum = 0.5),
            nn.LeakyReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features = 500, out_features = DETECTION_CLASSES, bias=True),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        x = self.conv(x)
        print(x.shape)
        x = self.fc(x)
        return x

In [3]:
class ClsTrainer(ResNet50):
    def __init__(self,
                lr,
                gamma,
                criterion):
        super(ClsTrainer, self).__init__()

        self.optim = torch.optim.Adam(super().parameters(), lr=lr)
        self.scheduler = torch.optim.lr_scheduler.ExponentialLR(self.optim, gamma=gamma)
        self.criterion = criterion
        self.epo_train_losses = []
        self.epo_val_losses = []

        self.train_losses = []
        self.val_losses = []

    def train_step(self, batch, target):
        self.train()
        self.optim.zero_grad()

        output = self.forward(batch)
        loss = self.criterion(output, target)

        loss.backward()
        self.optim.step()

        self.epo_train_losses.append(loss.item())

    def eval_step(self, batch, target):
        output = self.query(batch)
        loss = self.criterion(output, target)
        self.epo_val_losses.append(loss.item())

    def query(self, batch):
        self.eval()
        with torch.no_grad():
            output = self.forward(batch)

        return output

    def epoch_end(self):
        self.scheduler.step()

        self.train_losses.append(np.mean(self.epo_train_losses))
        self.val_losses.append(np.mean(self.epo_val_losses))

        self.epo_train_losses = []
        self.epo_val_losses = []

In [4]:
OBJ_SHAPE = (128,128)
class LabelTransform(torch.nn.Module):
    '''
    transforms CVAT data to proper tensor of labels
    '''
    def forward(self, Target):
        labels = []
        bboxes = []

        for data in Target.annotations.shapes:
            # лейблы в датасете начинаются с 9
            labels.append(data['label_id']-9)
            
            bboxes_raw = data['points'][-4:]
            bboxes.append([int(bboxes_raw[1]), int(bboxes_raw[0]), int(bboxes_raw[3]), int(bboxes_raw[2])])

        labels = torch.tensor(labels)
        labels = torch.nn.functional.one_hot(labels, DETECTION_CLASSES)

        return bboxes, labels

In [5]:
class ClsData:
    def __init__(self,
                dataset):
        self.len = len(dataset)
        self.dataset = dataset
        self.to_standart = transforms.Resize(OBJ_SHAPE)
        
    def __len__(self):
        return self.len

    def __getitem__(self, ind):
        image, (bboxes, labels) = self.dataset[ind]

        parts = []
        for box in bboxes:
            part = image[:,box[0]:box[2],box[1]:box[3]]
            part = self.to_standart(part)
            parts.append(part)

        parts = torch.stack(parts, dim=0)

        return parts.float(), labels.float()

In [6]:
with make_client(host="http://10.162.1.50:8080", credentials=('admin', 'qaedwsrf123')) as client:
    # get the dataset comprising all tasks for the Validation subset of project 12345
    dataset = ProjectVisionDataset(client, project_id=2,
                                  transform = transforms.ToTensor(),
                                  target_transform = LabelTransform())

train_size = len(dataset) - 50
val_size = 50

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])


print(f'train size: {len(train_dataset)}; val size: {len(val_dataset)}')

Server version '2.30.1' is not compatible with SDK version '2.32.0'. Some SDK functions may not work properly with this server. You can continue using this SDK, or you can try to update with 'pip install cvat-sdk'.


train size: 601; val size: 50


In [7]:
train_dataset = ClsData(train_dataset)
val_dataset = ClsData(val_dataset)

In [8]:
import ssl
ssl._create_default_https_context = ssl._create_stdlib_context

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

class_weights = torch.tensor([1.5, 1.0, 1.0, 1.0, 1.0, 1.0]).to(device)
classifier_criterion = torch.nn.BCELoss(weight = class_weights)

classifier = ClsTrainer(lr = 0.0005, gamma = 0.95, criterion = classifier_criterion).to(device)

summary(classifier, (3,128,128))

torch.Size([2, 1000])
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 64, 64, 64]           9,408
       BatchNorm2d-2           [-1, 64, 64, 64]             128
              ReLU-3           [-1, 64, 64, 64]               0
         MaxPool2d-4           [-1, 64, 32, 32]               0
            Conv2d-5           [-1, 64, 32, 32]           4,096
       BatchNorm2d-6           [-1, 64, 32, 32]             128
              ReLU-7           [-1, 64, 32, 32]               0
            Conv2d-8           [-1, 64, 32, 32]          36,864
       BatchNorm2d-9           [-1, 64, 32, 32]             128
             ReLU-10           [-1, 64, 32, 32]               0
           Conv2d-11          [-1, 256, 32, 32]          16,384
      BatchNorm2d-12          [-1, 256, 32, 32]             512
           Conv2d-13          [-1, 256, 32, 32]          16,384
      BatchNorm2d

In [9]:
# ~train loop
epoches = 5

for epo in tqdm(range(epoches)):
    for i in torch.randperm(len(train_dataset)):
        batch, targets = train_dataset[i]
        classifier.train_step(batch.to(device), targets.to(device))

    for image, targets in val_dataset:
        classifier.eval_step(batch.to(device), targets.to(device))

    classifier.epoch_end()

  0%|          | 0/5 [00:00<?, ?it/s]

torch.Size([1, 1000])


ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 1000])

In [ ]:
plt.plot(classifier.train_losses, label='train')
plt.plot(classifier.val_losses, label='eval')
plt.title("Классификатор")
plt.legend()
plt.grid(True)